[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maxencegu/python-datascience-m1/blob/main/cours/cm06_python_pratique_et_poo/cm06_cours.ipynb)

# **CM06 — Python Pratique & Programmation Orientée Objet**

**Python & Data Science · UPJV Amiens · M1 Économie**

---

Ce CM couvre les pratiques qui font passer du code qui fonctionne au code professionnel : gestion robuste des erreurs, organisation des fichiers, et modélisation avec des classes.

À l'issue de ce CM, vous saurez :
- organiser un projet Python (structure de dossiers, chemins) ;
- lire et écrire des fichiers avec le gestionnaire de contexte `with` ;
- gérer les erreurs avec `try / except / else / finally` ;
- définir des classes avec `__init__`, `self`, attributs et méthodes ;
- utiliser l'héritage pour spécialiser une classe ;
- choisir quand utiliser des classes vs des fonctions.

---
## 1. Structure d'un projet Python

Un projet bien organisé est plus facile à maintenir et à partager.

```
analyse-ventes/                ← dossier racine du projet
│
├── data/                      ← données brutes (ne pas modifier)
│   ├── ventes_2023.csv
│   └── clients.xlsx
│
├── output/                    ← résultats générés par le code
│   ├── graphiques/
│   └── rapports/
│
├── utils.py                   ← fonctions réutilisables
├── main.py                    ← script principal
├── pyproject.toml             ← dépendances (géré par uv)
└── README.md                  ← description du projet
```

**Règles** :
- `data/` = données d'entrée → jamais générées par le code
- `output/` = résultats → toujours regénérables par le code
- Ne jamais coder des chemins absolus (`/home/alice/projet/data/`) — utiliser des chemins relatifs

---
## 2. Chemins de fichiers

In [ ]:
import os
from pathlib import Path   # module moderne pour les chemins

# pathlib — syntaxe orientée objet, portable Windows/Mac/Linux
chemin_data = Path("data") / "ventes.csv"
print(chemin_data)                      # data/ventes.csv
print(chemin_data.suffix)               # .csv
print(chemin_data.stem)                 # ventes
print(chemin_data.parent)              # data

# Vérifier avant d'ouvrir
if chemin_data.exists():
    print("Fichier trouvé")
else:
    print("Fichier introuvable")

# Créer des dossiers
Path("output/graphiques").mkdir(parents=True, exist_ok=True)
print("Dossier output/graphiques créé")

In [ ]:
# os.path — ancienne syntaxe, encore très répandue
chemin = os.path.join("data", "ventes.csv")  # fonctionne sur tous les OS
print(chemin)
print(os.path.exists(chemin))

---
## 3. Lire et écrire des fichiers

### 3.1 Le gestionnaire de contexte `with`

In [ ]:
# Écrire un fichier texte
with open("notes.txt", "w", encoding="utf-8") as f:
    f.write("Alice : 15/20\n")
    f.write("Bob : 12/20\n")
    f.write("Charlie : 18/20\n")
# Le fichier est automatiquement fermé à la sortie du bloc with

# Lire le fichier entier
with open("notes.txt", "r", encoding="utf-8") as f:
    contenu = f.read()
print(contenu)

In [ ]:
# Lire ligne par ligne — efficace pour les gros fichiers
with open("notes.txt", "r", encoding="utf-8") as f:
    for ligne in f:
        ligne = ligne.strip()   # supprime \n en fin de ligne
        nom, note = ligne.split(" : ")
        print(f"{nom} → {note}")

In [ ]:
# Modes d'ouverture
# "r"  → lecture seule (par défaut)
# "w"  → écriture (écrase si le fichier existe)
# "a"  → ajout en fin de fichier
# "rb" / "wb" → lecture/écriture binaire (images, PDF)

# Ajouter une ligne à un fichier existant
with open("notes.txt", "a", encoding="utf-8") as f:
    f.write("Diane : 17/20\n")

# Vérifier
with open("notes.txt") as f:
    print(f.read())

---
## 4. Gestion des erreurs

Les erreurs sont inévitables. Une bonne gestion évite les crashs et produit des messages utiles.

### 4.1 Structure `try / except / else / finally`

In [ ]:
# Structure complète
try:
    # Code qui peut échouer
    resultat = 10 / 2
except ZeroDivisionError:
    # Exécuté si l'erreur spécifiée se produit
    print("Impossible de diviser par zéro")
else:
    # Exécuté SEULEMENT si aucune erreur
    print(f"Résultat : {resultat}")
finally:
    # Toujours exécuté (nettoyage)
    print("Fin du calcul")

### 4.2 Erreurs courantes à connaître

In [ ]:
# FileNotFoundError — fichier introuvable
try:
    with open("inexistant.csv") as f:
        contenu = f.read()
except FileNotFoundError:
    print("Fichier introuvable — vérifiez le chemin")

In [ ]:
# ValueError — valeur du mauvais format
def convertir_note(valeur):
    try:
        return float(valeur)
    except ValueError:
        print(f"'{valeur}' n'est pas une note valide")
        return None

print(convertir_note("15.5"))   # 15.5
print(convertir_note("abc"))    # message d'erreur
print(convertir_note(""))       # message d'erreur

In [ ]:
# KeyError — clé absente dans un dictionnaire
donnees = {"nom": "Alice", "age": 20}

try:
    ville = donnees["ville"]
except KeyError as e:
    print(f"Clé absente : {e}")

# Alternative sans try/except : la méthode get()
ville = donnees.get("ville", "Inconnue")   # valeur par défaut
print(ville)

In [ ]:
# IndexError — index hors limites
notes = [12, 15, 18]
try:
    print(notes[10])
except IndexError:
    print(f"Index invalide — la liste contient {len(notes)} éléments (0 à {len(notes)-1})")

In [ ]:
# TypeError — mauvais type
try:
    resultat = "10" + 5
except TypeError as e:
    print(f"Erreur de type : {e}")

In [ ]:
# Capturer plusieurs erreurs
def charger_donnee(source, cle):
    try:
        with open(source) as f:
            import json
            data = json.load(f)
        return data[cle]
    except FileNotFoundError:
        print(f"Fichier introuvable : {source}")
    except KeyError:
        print(f"Clé absente : {cle}")
    except json.JSONDecodeError:
        print(f"Le fichier {source} n'est pas du JSON valide")
    return None

resultat = charger_donnee("config.json", "version")
print(resultat)

### 4.3 Lire un traceback

Quand Python affiche une erreur, lisez **de bas en haut** :

```
Traceback (most recent call last):        ← début
  File "main.py", line 15, in <module>
    result = calculer(data)               ← appel
  File "main.py", line 8, in calculer
    return valeur / 0                     ← ligne du problème
ZeroDivisionError: division by zero       ← ← ← LIRE ICI EN PREMIER
```

1. **Type d'erreur** (`ZeroDivisionError`) → catégorie du problème
2. **Message** (`division by zero`) → détail
3. **Fichier et ligne** → où chercher

---
## 5. Programmation Orientée Objet (POO)

### 5.1 Pourquoi les classes ?

Une **classe** regroupe des données (attributs) et des comportements (méthodes) liés.

| Approche | Exemple |
|----------|--------|
| **Fonctions** | `calculer_surface(largeur, hauteur)` — bon pour une opération isolée |
| **Classe** | `Rectangle(largeur, hauteur)` avec `.surface()`, `.perimetre()` — bon quand données + comportements vont ensemble |

### 5.2 Définir une classe

In [ ]:
# Syntaxe de base
class Etudiant:
    # __init__ = constructeur, appelé à la création
    def __init__(self, prenom, nom, notes):
        self.prenom = prenom     # attribut d'instance
        self.nom = nom
        self.notes = notes

    # Méthode = fonction définie dans une classe
    def moyenne(self):
        if not self.notes:
            return 0
        return sum(self.notes) / len(self.notes)

    def mention(self):
        m = self.moyenne()
        if m >= 16:   return "Très bien"
        elif m >= 14: return "Bien"
        elif m >= 12: return "Assez bien"
        elif m >= 10: return "Passable"
        else:         return "Ajourné"

    # __str__ = représentation lisible (utilisée par print())
    def __str__(self):
        return f"{self.prenom} {self.nom} — Moyenne : {self.moyenne():.1f}/20 ({self.mention()})"

# Créer des instances
alice = Etudiant("Alice", "Dupont", [14, 16, 12, 18, 15])
bob   = Etudiant("Bob",   "Martin", [8, 11, 9, 12, 10])

print(alice)                      # utilise __str__
print(bob)
print(alice.prenom)               # accès à un attribut
print(alice.moyenne())             # appel d'une méthode

In [ ]:
# Les attributs peuvent être modifiés après création
alice.notes.append(17)
print(f"Après ajout d'une note : {alice.moyenne():.1f}")

# Plusieurs instances sont indépendantes
charlie = Etudiant("Charlie", "Durand", [17, 18, 19, 16, 20])
etudiants = [alice, bob, charlie]

for e in etudiants:
    print(e)

### 5.3 Attributs de classe vs attributs d'instance

In [ ]:
class Produit:
    taux_tva = 0.20          # attribut de CLASSE — partagé par toutes les instances
    nombre_produits = 0

    def __init__(self, nom, prix_ht):
        self.nom = nom                          # attribut d'INSTANCE — propre à chaque objet
        self.prix_ht = prix_ht
        Produit.nombre_produits += 1

    def prix_ttc(self):
        return self.prix_ht * (1 + Produit.taux_tva)

laptop = Produit("Laptop", 999)
souris = Produit("Souris", 29.99)

print(f"{laptop.nom} : {laptop.prix_ttc():.2f} € TTC")
print(f"{souris.nom} : {souris.prix_ttc():.2f} € TTC")
print(f"Total produits référencés : {Produit.nombre_produits}")

### 5.4 Héritage

In [ ]:
# Classe de base (parent)
class Animal:
    def __init__(self, nom, age):
        self.nom = nom
        self.age = age

    def se_presenter(self):
        return f"Je m'appelle {self.nom}, j'ai {self.age} ans"

    def faire_bruit(self):
        return "..."

# Classe dérivée (enfant) — hérite de Animal
class Chien(Animal):
    def __init__(self, nom, age, race):
        super().__init__(nom, age)   # appelle __init__ du parent
        self.race = race             # attribut propre à Chien

    def faire_bruit(self):           # surcharge (override)
        return "Woof !"

    def __str__(self):
        return f"{self.se_presenter()} — Race : {self.race} — {self.faire_bruit()}"

class Chat(Animal):
    def faire_bruit(self):
        return "Miaou !"

# Utilisation
rex    = Chien("Rex", 3, "Labrador")
minou  = Chat("Minou", 5)

print(rex)
print(f"{minou.nom} dit : {minou.faire_bruit()}")

# isinstance() vérifie l'appartenance
print(isinstance(rex, Chien))    # True
print(isinstance(rex, Animal))   # True — Chien est un Animal
print(isinstance(rex, Chat))     # False

In [ ]:
# Exemple data science : hiérarchie de modèles
class ModeleBase:
    def __init__(self, nom, donnees):
        self.nom = nom
        self.donnees = donnees
        self.est_entraine = False

    def entrainer(self):
        self.est_entraine = True
        print(f"{self.nom} : entraînement terminé")

    def predire(self, x):
        raise NotImplementedError("Chaque modèle doit implémenter predire()")

class RegressionLineaire(ModeleBase):
    def __init__(self, donnees):
        super().__init__("Régression Linéaire", donnees)
        self.pente = None
        self.intercept = None

    def entrainer(self):
        # Simulation : en pratique on utilise scikit-learn
        self.pente = 2.5
        self.intercept = 10
        super().entrainer()

    def predire(self, x):
        if not self.est_entraine:
            raise RuntimeError("Modèle non entraîné")
        return self.pente * x + self.intercept

modele = RegressionLineaire([1, 2, 3, 4, 5])
modele.entrainer()
print(f"Prédiction pour x=7 : {modele.predire(7)}")

### 5.5 Quand utiliser des classes ?

| Situation | Recommandation |
|-----------|----------------|
| Opération simple sur des données | **Fonction** |
| Pipeline de traitement → plusieurs étapes | **Fonctions** ordonnées |
| Données + comportements qui vont ensemble | **Classe** |
| Plusieurs variantes d'un même concept | **Héritage** |
| État qui évolue au fil du temps | **Classe** |

**Règle pratique** : commencez par des fonctions. Créez une classe quand vous vous retrouvez à passer les mêmes données à plusieurs fonctions.

---
## 6. Exemple complet : Gestionnaire de données

Combinaison de tout ce que nous avons vu : fichiers, gestion d'erreurs, classe.

In [ ]:
import pandas as pd
import json
from pathlib import Path

class GestionnaireDonnees:
    """Charge, valide et analyse un fichier de données."""

    def __init__(self, chemin_fichier):
        self.chemin = Path(chemin_fichier)
        self.df = None
        self.erreurs = []

    def charger(self):
        """Charge le fichier CSV."""
        try:
            self.df = pd.read_csv(self.chemin)
            print(f"✅ {len(self.df)} lignes chargées depuis {self.chemin.name}")
            return True
        except FileNotFoundError:
            self.erreurs.append(f"Fichier introuvable : {self.chemin}")
            print(f"❌ Fichier introuvable : {self.chemin}")
            return False
        except pd.errors.EmptyDataError:
            self.erreurs.append("Fichier vide")
            print("❌ Le fichier est vide")
            return False

    def statistiques(self, colonne):
        """Retourne les statistiques d'une colonne numérique."""
        if self.df is None:
            raise RuntimeError("Données non chargées — appelez charger() d'abord")
        try:
            return {
                "moyenne": self.df[colonne].mean(),
                "median":  self.df[colonne].median(),
                "min":     self.df[colonne].min(),
                "max":     self.df[colonne].max(),
            }
        except KeyError:
            raise KeyError(f"Colonne '{colonne}' absente. Disponibles : {list(self.df.columns)}")

    def exporter_rapport(self, chemin_sortie):
        """Exporte un rapport JSON."""
        rapport = {
            "source": str(self.chemin),
            "lignes": len(self.df) if self.df is not None else 0,
            "colonnes": list(self.df.columns) if self.df is not None else [],
            "erreurs": self.erreurs
        }
        with open(chemin_sortie, "w", encoding="utf-8") as f:
            json.dump(rapport, f, indent=2, ensure_ascii=False)
        print(f"Rapport exporté : {chemin_sortie}")

# Utilisation
gestionnaire = GestionnaireDonnees("ventes.csv")
if gestionnaire.charger():
    try:
        stats = gestionnaire.statistiques("ca")
        print(f"\nCA — Moyenne : {stats['moyenne']:.0f} € | Max : {stats['max']:.0f} €")
    except KeyError as e:
        print(f"Colonne introuvable : {e}")
    gestionnaire.exporter_rapport("rapport.json")

---
## 7. Récapitulatif

```python
# Fichiers
with open("fichier.txt", "r", encoding="utf-8") as f:
    contenu = f.read()

# Gestion d'erreurs
try:
    resultat = operation_risquee()
except TypeErreur as e:
    print(f"Erreur : {e}")
else:
    print(resultat)    # si pas d'erreur
finally:
    nettoyage()        # toujours

# Classe
class MonObjet:
    def __init__(self, param):
        self.param = param

    def ma_methode(self):
        return self.param

    def __str__(self):
        return f"MonObjet({self.param})"

# Héritage
class Enfant(Parent):
    def __init__(self, param_parent, param_enfant):
        super().__init__(param_parent)   # appel parent
        self.param_enfant = param_enfant
```

| Erreur | Situation typique |
|--------|------------------|
| `FileNotFoundError` | Fichier/chemin incorrect |
| `ValueError` | Conversion de type échoue |
| `KeyError` | Clé absente dans un dict |
| `IndexError` | Index hors limites |
| `TypeError` | Types incompatibles |

### Prochain CM
**CM07 — Git, GitHub & Outils modernes** : contrôle de version, branches, Pull Requests, variables d'environnement (.env), Ruff (linter) et UV (gestionnaire de dépendances rapide).